# Otto Scents: YOLOv8 Large (v8l) Training
This notebook uses the **Large (L)** model to improve detection accuracy for your A, B, C, D bottle caps while keeping the misplacement logic.

### 1. Install Requirements

In [ ]:
!pip install ultralytics roboflow pyyaml

### 2. Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="9oRKwXj2MdugvomIxwFW")
project = rf.workspace("jc-jade").project("perfume-letters")
version = project.version(1)
dataset = version.download("yolov8")

### 3. Fix Dataset Paths
This cell fixes the 'images not found' error by creating a clean absolute-path YAML file.

In [ ]:
import yaml
import os

base_path = dataset.location
train_path = ""
val_path = ""

for folder in ['train', 'valid', 'val']:
    full_path = os.path.join(base_path, folder, 'images')
    if os.path.exists(full_path):
        if folder == 'train':
            train_path = full_path
        else:
            val_path = full_path

data_fixed = {
    'path': base_path,
    'train': train_path,
    'val': val_path,
    'names': {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
}

fixed_yaml_path = os.path.join(base_path, 'fixed_data.yaml')
with open(fixed_yaml_path, 'w') as f:
    yaml.dump(data_fixed, f, default_flow_style=False)

print(f"✅ Fixed YAML created at: {fixed_yaml_path}")

### 4. Train the Model (YOLOv8 Large)
Training the Large model for significantly better accuracy than the Nano version.

In [ ]:
from ultralytics import YOLO

# Using the Large model
model = YOLO('yolov8l.pt')

results = model.train(
    data=os.path.join(dataset.location, 'fixed_data.yaml'),
    epochs=100,
    imgsz=640,
    batch=8,  # Large models use more memory, batch 8 is safer for Colab T4
    plots=True
)

### 5. Validation & Export

In [ ]:
from google.colab import files
from ultralytics import YOLO

# Load best weights from training
best_path = 'runs/detect/train/weights/best.pt'
if not os.path.exists(best_path):
    # Fallback in case of multiple runs
    import glob
    run_dirs = glob.glob('runs/detect/train*')
    latest_run = max(run_dirs, key=os.path.getmtime)
    best_path = os.path.join(latest_run, 'weights/best.pt')

best_model = YOLO(best_path)
best_model.export(format='onnx')

print(f"Downloading best weights from {best_path}...")
files.download(best_path)
files.download(best_path.replace('.pt', '.onnx'))